# Distance-dependent GP tests for four ATAC-derived modalities

This notebook shows how to run the same fixed-noise Gaussian-process (GP) likelihood-ratio test on four processed inputs from the ligand-distance analysis:

1. called ATAC peaks;
2. fixed 5-kb ATAC bins;
3. chromVAR motif-deviation scores; and
4. Signac ATAC gene-activity scores.

The worked example uses **WNT5A**. Each input has already been pseudobulked across cells in spatial distance bins and contains the distance-bin midpoints (`X`), feature means (`means`), standard errors (`sems`), and the calibrated fixed-noise variance (`mean_sem_sq`). The peak input uses six 50-µm bins; the other inputs use twelve 25-µm bins. All four inputs were filtered to features detected in more than 5% of cells before GP testing.


## Statistical test

For each feature, the test compares a zero-mean GP with a constant (Bias) covariance kernel against a zero-mean GP with an RBF covariance kernel. The Gaussian observation-noise variance is fixed, rather than optimized, at the mean squared pseudobulk SEM across distance bins for that feature: $\sigma_f^2 = B^{-1}\sum_b SEM_{bf}^2$. This SEM calibration prevents features with noisier pseudobulks from receiving the same effective weight as precisely estimated features.

The RBF fit uses three deterministic optimization starts with initial lengthscales of 10, 50, and 200 µm; the best likelihood is retained, with at most 1,000 L-BFGS-B iterations per fit. These are three starts, not three additional random restarts. The primary likelihood-ratio p-value uses the approximate boundary reference $0.5\chi_0^2 + 0.5\chi_1^2$, because the constant model is approached as the RBF lengthscale tends to infinity. Benjamini-Hochberg correction is applied separately within each perturbation-by-modality screen at 5% FDR. Ordinary $\chi_1^2$ and Benjamini-Yekutieli results are retained by the runner as sensitivity analyses.

An optional minimum RBF lengthscale can exclude changes sharper than a prespecified spatial scale. The production screens reproduced here were unconstrained. For the previously used constrained sensitivity analysis on 25-µm spatial bins, set `MIN_LENGTHSCALE_UM = 25.0`.


## 1. Locate the project and configure the run

The GP dependencies are isolated in `gpclust_spatial/.venv_gpclust`. The notebook launches that interpreter explicitly, so its own kernel only needs NumPy and pandas for input inspection and result display. Leave `RUN_SCREENS = False` to inspect the inputs, generated commands, and completed production results without refitting. Set it to `True` to create new results under `results/notebook_multimodal_atac_gp/`.


In [ ]:
from pathlib import Path
import json
import subprocess

import numpy as np
import pandas as pd


def find_workspace(start=Path.cwd().resolve()):
    for candidate in (start, *start.parents):
        runner = candidate / "gpclust_spatial" / "run_spatial_lrt.py"
        if runner.is_file() and (candidate / "results").is_dir():
            return candidate
    raise FileNotFoundError(
        "Could not find a parent containing gpclust_spatial/run_spatial_lrt.py and results/."
    )


WORKSPACE = find_workspace()
GP_RUNNER = WORKSPACE / "gpclust_spatial" / "run_spatial_lrt.py"
GP_PYTHON = WORKSPACE / "gpclust_spatial" / ".venv_gpclust" / "Scripts" / "python.exe"

TARGET = "WNT5A"
FDR = 0.05
WORKERS = 6
CHUNK_SIZE = 25
MAX_OPTIMIZER_ITERS = 1000
RBF_START_LENGTHSCALES_UM = (10.0, 50.0, 200.0)
MIN_LENGTHSCALE_UM = None  # Set to 25.0 for the constrained sensitivity analysis.

RUN_SCREENS = False
DIAGNOSTIC_MAX_FEATURES = None  # For a smoke test, use an integer such as 100.
OUTPUT_ROOT = WORKSPACE / "results" / "notebook_multimodal_atac_gp" / TARGET

assert GP_RUNNER.is_file(), GP_RUNNER
assert GP_PYTHON.is_file(), GP_PYTHON
print(f"Workspace: {WORKSPACE}")
print(f"GP interpreter: {GP_PYTHON}")


## 2. Register the four modality inputs

The `reference_output` entries point to the completed production runs and are used only for display when `RUN_SCREENS` is false. New fits always go to `OUTPUT_ROOT`; the notebook does not overwrite production results. The shared runner uses the generic field name `gene` for feature identifiers in its output table, including peaks, genomic bins, and motifs.


In [ ]:
target_slug = TARGET.lower()

MODALITIES = {
    "ATAC peaks": {
        "input": WORKSPACE / "results" / "wnt5a_atac_peak_pseudobulk_logcpm_gp_assigned_50um" / "wnt5a_atac_peak_pseudobulk_logcpm_inputs.npz",
        "expected_bins": 6,
        "spatial_bin_width_um": 50,
        "feature_type": "called peak",
        "reference_output": WORKSPACE / "results" / "wnt5a_atac_peak_pseudobulk_logcpm_gp_assigned_50um" / "01_spatial_peak_screen",
    },
    "ATAC 5-kb bins": {
        "input": WORKSPACE / "results" / "atac_ligand_distance_25um" / "snapatac2_tiles_5000bp" / "conditions" / target_slug / "gp_inputs.npz",
        "expected_bins": 12,
        "spatial_bin_width_um": 25,
        "feature_type": "5-kb genomic bin",
        "reference_output": WORKSPACE / "results" / "atac_ligand_distance_25um" / "snapatac2_tiles_5000bp" / "conditions" / target_slug / "gp_lrt",
    },
    "chromVAR scores": {
        "input": WORKSPACE / "results" / "chromvar_all_ligands_gp_assigned_25um" / "datasets" / TARGET / f"{target_slug}_chromvar_inputs.npz",
        "expected_bins": 12,
        "spatial_bin_width_um": 25,
        "feature_type": "motif deviation score",
        "reference_output": WORKSPACE / "results" / "chromvar_all_ligands_gp_assigned_25um" / "datasets" / TARGET / "01_spatial_motif_screen",
    },
    "ATAC gene activity": {
        "input": WORKSPACE / "results" / "atac_ligand_distance_25um" / "signac_gene_activity" / "conditions" / target_slug / "gp_inputs.npz",
        "expected_bins": 12,
        "spatial_bin_width_um": 25,
        "feature_type": "Signac gene activity",
        "reference_output": WORKSPACE / "results" / "atac_ligand_distance_25um" / "signac_gene_activity" / "conditions" / target_slug / "gp_lrt",
    },
}

if TARGET != "WNT5A":
    raise ValueError(
        "This worked four-modality example uses the available WNT5A peak input. "
        "The 5-kb-bin, chromVAR, and gene-activity path templates can be reused for other targets."
    )


In [ ]:
REQUIRED_ARRAYS = {"X", "means", "sems", "mean_sem_sq", "gene_names", "gene_indices"}


def inspect_gp_input(label, spec):
    path = spec["input"]
    if not path.is_file():
        raise FileNotFoundError(f"Missing {label} input: {path}")

    with np.load(path, allow_pickle=False) as data:
        missing = REQUIRED_ARRAYS.difference(data.files)
        if missing:
            raise ValueError(f"{label} is missing arrays: {sorted(missing)}")

        X = data["X"]
        means = data["means"]
        sems = data["sems"]
        mean_sem_sq = data["mean_sem_sq"]
        feature_ids = data["gene_names"]

        if X.shape != (spec["expected_bins"], 1):
            raise ValueError(f"Unexpected X shape for {label}: {X.shape}")
        if means.shape != sems.shape or means.shape[0] != X.shape[0]:
            raise ValueError(f"Incompatible mean/SEM shapes for {label}")
        if means.shape[1] != feature_ids.size or mean_sem_sq.shape != (feature_ids.size,):
            raise ValueError(f"Incompatible feature arrays for {label}")

        calibrated_noise = np.mean(np.square(sems), axis=0)
        if not np.allclose(mean_sem_sq, calibrated_noise, rtol=1e-8, atol=1e-12):
            raise ValueError(f"mean_sem_sq is not mean(sems ** 2) for {label}")

        detection_min = float(np.min(data["detection"])) if "detection" in data.files else np.nan
        normalization = str(data["normalization_method"]) if "normalization_method" in data.files else "score mean"
        return {
            "modality": label,
            "features": feature_ids.size,
            "distance bins": X.shape[0],
            "bin width (um)": spec["spatial_bin_width_um"],
            "midpoint range (um)": f"{X[0, 0]:g}-{X[-1, 0]:g}",
            "minimum detection": detection_min,
            "normalization": normalization,
        }


input_summary = pd.DataFrame(
    [inspect_gp_input(label, spec) for label, spec in MODALITIES.items()]
).set_index("modality")
input_summary


### Interpretation of the processed values

For peaks and 5-kb bins, fragments were summed within each spatial pseudobulk, library-size normalized, and transformed as `log1p(10000 * feature_count / total_count)`. Signac gene-activity counts were processed analogously using the gene-activity library total. The SEMs were propagated from cell-level count and library-size variation with the count/library-size covariance retained. chromVAR inputs instead contain the mean and SEM of per-cell motif deviation scores in each distance bin. GP testing begins from these processed matrices; it does not renormalize the observations.


## 3. Build and optionally execute the GP screens

A diagnostic subset is useful only for checking that the environment and paths work. Because FDR correction on a first-N subset is not the prespecified full screen, do not report diagnostic q-values. Use `DIAGNOSTIC_MAX_FEATURES = None` for final results. Each final screen gets its own output directory, so BH correction is performed independently for every modality and perturbation.


In [ ]:
def output_directory(label):
    slug = label.lower().replace(" ", "_").replace("-", "")
    subset = "full" if DIAGNOSTIC_MAX_FEATURES is None else f"first_{DIAGNOSTIC_MAX_FEATURES}"
    constraint = "unconstrained" if MIN_LENGTHSCALE_UM is None else f"min_l{MIN_LENGTHSCALE_UM:g}um"
    return OUTPUT_ROOT / slug / f"gp_lrt_{subset}_{constraint}"


def build_gp_command(label, spec):
    command = [
        str(GP_PYTHON),
        str(GP_RUNNER),
        "--input", str(spec["input"]),
        "--output", str(output_directory(label)),
        "--fdr", str(FDR),
        "--rbf-start-lengthscales", *map(str, RBF_START_LENGTHSCALES_UM),
        "--max-optimizer-iters", str(MAX_OPTIMIZER_ITERS),
        "--workers", str(WORKERS),
        "--parallel-backend", "process",
        "--chunk-size", str(CHUNK_SIZE),
        "--expected-bins", str(spec["expected_bins"]),
    ]
    if MIN_LENGTHSCALE_UM is not None:
        command.extend(["--rbf-lengthscale-minimum", str(MIN_LENGTHSCALE_UM)])
    if DIAGNOSTIC_MAX_FEATURES is not None:
        command.extend(["--diagnostic-max-genes", str(DIAGNOSTIC_MAX_FEATURES)])
    return command


def run_gp_screen(label, spec):
    output = output_directory(label)
    config_path = output / "run_config.json"
    if config_path.is_file():
        config = json.loads(config_path.read_text(encoding="utf-8"))
        if config.get("status") == "completed":
            print(f"Reusing completed screen: {output}")
            return output
    if output.exists() and any(output.iterdir()):
        raise FileExistsError(f"Refusing to overwrite an incomplete/nonempty output: {output}")
    subprocess.run(build_gp_command(label, spec), cwd=WORKSPACE, check=True)
    return output


command_table = pd.DataFrame(
    {
        "modality": label,
        "command": subprocess.list2cmdline(build_gp_command(label, spec)),
    }
    for label, spec in MODALITIES.items()
).set_index("modality")

if RUN_SCREENS:
    for modality_label, modality_spec in MODALITIES.items():
        print(f"\nRunning {TARGET} / {modality_label}")
        run_gp_screen(modality_label, modality_spec)
else:
    print("RUN_SCREENS is False; commands were built but not executed.")

command_table


## 4. Read and compare the results

When new screens were not requested, this section reads the completed production outputs listed above. The primary columns are `likelihood_ratio_statistic`, boundary-mixture `p_value`, `fdr_q_value`, and `significant_fdr`. A fitted lengthscale longer than the observed distance span is flagged because that fit is effectively close to the constant-model boundary.


In [ ]:
def result_directory(label, spec):
    return output_directory(label) if RUN_SCREENS else spec["reference_output"]


def load_results(label, spec):
    directory = result_directory(label, spec)
    table_path = directory / "spatial_lrt_results.tsv"
    config_path = directory / "run_config.json"
    if not table_path.is_file() or not config_path.is_file():
        raise FileNotFoundError(f"Missing completed GP results in {directory}")

    table = pd.read_csv(table_path, sep="\t")
    config = json.loads(config_path.read_text(encoding="utf-8"))
    table.insert(0, "modality", label)

    if label == "chromVAR scores":
        with np.load(spec["input"], allow_pickle=False) as data:
            motif_name = dict(zip(data["motif_ids"].astype(str), data["motif_names"].astype(str)))
        table.insert(2, "display_feature", table["gene"].map(motif_name).fillna(table["gene"]))
    else:
        table.insert(2, "display_feature", table["gene"])

    return table, config, directory


RESULTS = {}
run_rows = []
for modality_label, modality_spec in MODALITIES.items():
    table, config, directory = load_results(modality_label, modality_spec)
    RESULTS[modality_label] = table
    run_rows.append(
        {
            "modality": modality_label,
            "tested": len(table),
            "fit failures": int((~table["fit_success"]).sum()),
            "significant at 5% FDR": int(table["significant_fdr"].sum()),
            "minimum lengthscale (um)": config.get("rbf_lengthscale_minimum_um"),
            "result directory": str(directory),
        }
    )

run_summary = pd.DataFrame(run_rows).set_index("modality")
run_summary


In [ ]:
TOP_N = 10
DISPLAY_COLUMNS = [
    "modality",
    "gene",
    "display_feature",
    "detection_fraction",
    "expression_range",
    "likelihood_ratio_statistic",
    "p_value",
    "fdr_q_value",
    "rbf_lengthscale_um",
    "significant_fdr",
]

top_hits = pd.concat(
    [
        table.loc[table["fit_success"]]
        .sort_values(["p_value", "likelihood_ratio_statistic"], ascending=[True, False])
        .head(TOP_N)[DISPLAY_COLUMNS]
        for table in RESULTS.values()
    ],
    ignore_index=True,
)
top_hits


## Extending the screen to other ligands

The 5-kb-bin and gene-activity inputs use `conditions/<lowercase-target>/gp_inputs.npz`; chromVAR uses `datasets/<UPPERCASE-TARGET>/<lowercase-target>_chromvar_inputs.npz`. Construct a fresh `MODALITIES` dictionary for each target and call `run_gp_screen` separately so every ligand receives its own multiple-testing correction. The current normalized called-peak example is WNT5A-specific; create the corresponding peak-level pseudobulk NPZ before adding another ligand to that modality. Negative-control datasets can be run through the same path templates and test settings.

For final analyses, keep `DIAGNOSTIC_MAX_FEATURES = None`, record the generated `run_config.json` and input SHA-256 hashes, inspect fit failures and boundary flags, and treat the constrained minimum-lengthscale screen as a labeled sensitivity analysis rather than silently mixing it with the unconstrained primary results.
